In [27]:
import pandas as pd
import os
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [12]:
X_train = pd.read_csv('../data/interim/X_train.csv')
y_train = pd.read_csv('../data/interim/y_train.csv')
X_val = pd.read_csv('../data/interim/X_val.csv')
y_val = pd.read_csv('../data/interim/y_val.csv')

# 1. Preprocessing
## 1.1 Missing Values

In [13]:
print("missing values:")
print(X_train.isnull().sum())

missing values:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age            137
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          552
Embarked         2
dtype: int64


**Observations**
- We discussed in a previous phase that we will drop **Cabin** for the following reasons:
    - 77.42% of **Cabin** values are missing.
    - These missing values indicate according to the previous findings that almost belong to rows where the **Pclass** is 3.
- For **Embarked** there are only two missing values, we can fill them with the most often value in that column.
- 19.21% of **Age** values are missing. Since it is numerical feature we can fill the missing values with the median better than the mean sinve there are some outliers (80 yo) which may affect the mean.

In [14]:
# --- 1. Drop Cabin ---
X_train = X_train.drop(columns=['Cabin'])
X_val = X_val.drop(columns=['Cabin'])

# --- 2. Fill Embarked with the mode (most frequent value), computed from X_train only ---
embarked_mode = X_train['Embarked'].mode()[0]
print(f"Embarked mode (from train): {embarked_mode}")

X_train['Embarked'] = X_train['Embarked'].fillna(embarked_mode)
X_val['Embarked'] = X_val['Embarked'].fillna(embarked_mode)

# --- 3. Fill Age with the median, computed from X_train only ---
age_median = X_train['Age'].median()
print(f"Age median (from train): {age_median}")

X_train['Age'] = X_train['Age'].fillna(age_median)
X_val['Age'] = X_val['Age'].fillna(age_median)

# --- sanity check ---
print(X_train.isnull().sum())
print(X_val.isnull().sum())

Embarked mode (from train): S
Age median (from train): 28.5
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## 1.2 Feature Engineering
### 1.2.1 Name
As discussed in the previous notebook, we will extract only the title (Mr., Mrs.,...) from the **Name** since it is the only information we need rather than the full name.

Looking at "Braund, Mr. Owen Harris". The title sits between two specific characters:

- Right before it: a comma and a space (", ")
- Right after it: a period (".")

In [15]:
X_train['Title'] = X_train['Name'].str.extract(r',\s*([^\.]+)\.')
X_val['Title'] = X_val['Name'].str.extract(r',\s*([^\.]+)\.')

print(X_train['Title'].value_counts())

Title
Mr          412
Miss        141
Mrs         107
Master       31
Dr            6
Rev           5
Mlle          2
Col           2
Major         1
Lady          1
Sir           1
Ms            1
Jonkheer      1
Don           1
Name: count, dtype: int64


**Observations**
- We notice that there are some titles that have few samples.
- We can map *Mlle* and *Ms* to *Miss*.

**Decisions**
- We map *Mlle* and *Ms* to *Miss* because Mlle and Ms are direct linguistic equivalents of Miss, not distinct status/rank titles like the rest, so they're merged rather than grouped into Other.
- We regroup *Dr*, *Rev*, *Col*, *Major*, *Lady*, *Sir*, *Jonkheer* and *Don* to one group named **Other**.

In [16]:
title_mapping = {
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
}

# everything not explicitly mapped above falls into 'Other'
def map_title(title):
    return title_mapping.get(title, 'Other')

X_train['Title'] = X_train['Title'].apply(map_title)
X_val['Title'] = X_val['Title'].apply(map_title)

print(X_train['Title'].value_counts())
print(X_val['Title'].value_counts())

Title
Mr        412
Miss      144
Mrs       107
Master     31
Other      18
Name: count, dtype: int64
Title
Mr        105
Miss       41
Mrs        18
Master      9
Other       6
Name: count, dtype: int64


### 1.2.2 Ticket
We found out previously that a **Ticket** can be shared across many people that do not necessarily belong to the same family, and the **Fare** is divide on the total number of that group. For this, we decided to create two new features, **Groupe_Size** which includes the total number of the group per **Ticket** and another feature named **Fare_per_person**, it get the exact amount payed by each member of that group by dividing the **Fare** on the **Groupe_Size**.

Group_Size was computed using Ticket values from both X_train and X_val combined, since ticket group membership is a fixed historical fact (not derived from Survived), unlike statistics such as Age's median which must be train-only.

In [17]:
# Combine Ticket values from both train and val (structural fact only, no Survived involved)
all_tickets = pd.concat([X_train['Ticket'], X_val['Ticket']])
ticket_counts = all_tickets.value_counts()

X_train['Group_Size'] = X_train['Ticket'].map(ticket_counts)
X_val['Group_Size'] = X_val['Ticket'].map(ticket_counts)

# Now compute Fare_per_person using the raw Fare (not yet scaled)
X_train['Fare_per_person'] = X_train['Fare'] / X_train['Group_Size']
X_val['Fare_per_person'] = X_val['Fare'] / X_val['Group_Size']

print(X_train[['Ticket', 'Group_Size', 'Fare', 'Fare_per_person']].head(10))

       Ticket  Group_Size      Fare  Fare_per_person
0        1601           7   56.4958         8.070829
1      239854           1    0.0000         0.000000
2    PC 17483           1  221.7792       221.779200
3      392091           1    9.3500         9.350000
4  C.A. 31921           3   26.2500         8.750000
5        8475           1    8.4333         8.433300
6        1601           7   56.4958         8.070829
7    PC 17757           4  227.5250        56.881250
8      367228           1    7.7500         7.750000
9      345763           1   18.0000        18.000000


## 1.3 Encoding Categorical Features

**The core problem:** most ML models (Logistic Regression, SVMs, even most tree implementations under the hood) work with numbers, not text. Columns like Sex ("male"/"female") or Embarked ("C"/"Q"/"S") are strings — the model literally can't do math on them. So we need a way to turn categories into numeric representations. That's what "encoding" means.
### 1.3.1 One-Hot Encoding

Creates a separate binary (0/1) column for each category.Best for low-cardinality categoricals with no natural order (nominal data) — like Sex or Embarked. There's no sense in which "Cherbourg > Southampton" — they're just different, unordered labels.

#### 1.3.1.1 Embarked
Here we will create 3 columns Embarked_C, Embarked_Q and Embarked_S. When Embarked = C so Embarked_C = 1, and the two rest columns will be set to 0. 


In [18]:
# One-hot encode Embarked
X_train = pd.get_dummies(X_train, columns=['Embarked'], prefix='Embarked')
X_val = pd.get_dummies(X_val, columns=['Embarked'], prefix='Embarked')

print(X_train.columns.tolist())
X_train.head()

['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Title', 'Group_Size', 'Fare_per_person', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Title,Group_Size,Fare_per_person,Embarked_C,Embarked_Q,Embarked_S
0,693,3,"Lam, Mr. Ali",male,28.5,0,0,1601,56.4958,Mr,7,8.070829,False,False,True
1,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,28.5,0,0,239854,0.0000,Mr,1,0.000000,False,False,True
2,528,1,"Farthing, Mr. John",male,28.5,0,0,PC 17483,221.7792,Mr,1,221.779200,False,False,True
3,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Mrs,1,9.350000,False,False,True
4,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Mrs,3,8.750000,False,False,True


In [19]:
print(X_val.columns.tolist())
X_val.head()

['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Title', 'Group_Size', 'Fare_per_person', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Title,Group_Size,Fare_per_person,Embarked_C,Embarked_Q,Embarked_S
0,566,3,"Davies, Mr. Alfred J",male,24.0,2,0,A/4 48871,24.1500,Mr,2,12.0750,False,False,True
1,161,3,"Cribb, Mr. John Hatfield",male,44.0,0,1,371362,16.1000,Mr,1,16.1000,False,False,True
2,554,3,"Leeni, Mr. Fahim (""Philip Zenni"")",male,22.0,0,0,2620,7.2250,Mr,1,7.2250,True,False,False
3,861,3,"Hansen, Mr. Claus Peter",male,41.0,2,0,350026,14.1083,Mr,1,14.1083,False,False,True
4,242,3,"Murphy, Miss. Katherine ""Kate""",female,28.5,1,0,367230,15.5000,Miss,2,7.7500,False,True,False


#### 1.3.1.2 Title

In the feature engineering we extracted only the **Title** from **Name** and we got 5-category column (Mr, Miss, Mrs, Master, Other).

get_dummies has a risk (the category-mismatch issue between train/val/test), therefore we will use sklearn's OneHotEncoder.

In [21]:
# columns to one-hot encode
cat_cols = ['Title']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# fit ONLY on X_train
encoder.fit(X_train[cat_cols])

# transform both train and val using the SAME fitted encoder
train_encoded = encoder.transform(X_train[cat_cols])
val_encoded = encoder.transform(X_val[cat_cols])

# turn the output back into a DataFrame with proper column names
encoded_cols = encoder.get_feature_names_out(cat_cols)

train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_cols, index=X_train.index)
val_encoded_df = pd.DataFrame(val_encoded, columns=encoded_cols, index=X_val.index)

# drop original categorical columns and join the new encoded ones
X_train = X_train.drop(columns=cat_cols).join(train_encoded_df)
X_val = X_val.drop(columns=cat_cols).join(val_encoded_df)

print(X_train.columns.tolist())
X_train.head()

['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Group_Size', 'Fare_per_person', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Other']


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Group_Size,Fare_per_person,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Other
0,693,3,"Lam, Mr. Ali",male,28.5,0,0,1601,56.4958,7,8.070829,False,False,True,0.0,0.0,1.0,0.0,0.0
1,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,28.5,0,0,239854,0.0000,1,0.000000,False,False,True,0.0,0.0,1.0,0.0,0.0
2,528,1,"Farthing, Mr. John",male,28.5,0,0,PC 17483,221.7792,1,221.779200,False,False,True,0.0,0.0,1.0,0.0,0.0
3,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,1,9.350000,False,False,True,0.0,0.0,0.0,1.0,0.0
4,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,3,8.750000,False,False,True,0.0,0.0,0.0,1.0,0.0


### 1.3.2 Binary/Label Encoding
#### 1.3.2.1 Sex
since Sex has exactly 2 categories, a single 0/1 column loses no information (unlike squeezing a 3-category unordered column like Embarked into one numeric column, which would falsely imply an order).

In [22]:
X_train['Sex'] = X_train['Sex'].map({'male': 0, 'female': 1})
X_val['Sex'] = X_val['Sex'].map({'male': 0, 'female': 1})

print(X_train['Sex'].value_counts())
print(X_val['Sex'].value_counts())

Sex
0    459
1    253
Name: count, dtype: int64
Sex
0    118
1     61
Name: count, dtype: int64


### 1.3.3 Ordinal Encoding
**Pclass** is already oridnal and based on previous results:
- Class 1: 0.649
- Class 2: 0.447
- Class 3: 0.243

class 1→2 drops by about 0.20, and class 2→3 drops by about 0.20 too.the gaps are roughly similar, leaving Pclass as a raw number is a reasonably safe simplification for a linear model.

## 1.4 Dropping Uneeded Columns

In [24]:
cols_to_drop = ['Name', 'Ticket', 'PassengerId']

X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)

print(X_train.columns.tolist())

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Group_Size', 'Fare_per_person', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Other']


## 1.5 Features Scaling
Scaled (continuous or wider-range numeric, to prevent magnitude from distorting linear models):

- Age — continuous, range ~0–80. Large enough spread that it could dominate a weighted-sum model like Logistic Regression if left unscaled.
- Fare — continuous, range ~0–500+. Same reasoning, even more extreme range.
- Fare_per_person — same nature as Fare, just adjusted for group size; scaled for consistency.
- SibSp, Parch, Group_Size — technically small-range counts (0–8ish), so the "magnitude domination" risk is much milder than Age/Fare — but you chose to scale them anyway for a simple, consistent rule ("scale every non-binary numeric feature") rather than judgment-calling each one individually.

Not scaled (already binary 0/1, or deliberately left ordinal):

- Sex — binary (0/1). Already the smallest possible numeric scale; nothing to "spread out."
- Embarked_C, Embarked_Q, Embarked_S — one-hot, each strictly 0/1.
- Title_Mr, Title_Miss, Title_Mrs, Title_Master, Title_Other — one-hot, each strictly 0/1.
- Pclass — technically numeric (1/2/3), but deliberately left raw and unscaled since you confirmed the survival-rate gaps between classes were roughly even (~0.20 each), making it reasonably safe to treat as a small ordinal scale without needing standardization.

In [26]:
cols_to_scale = ['Age', 'Fare', 'Fare_per_person', 'SibSp', 'Parch', 'Group_Size']

scaler = StandardScaler()

# fit ONLY on X_train
scaler.fit(X_train[cols_to_scale])

# transform both, using the same fitted scaler
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_val[cols_to_scale] = scaler.transform(X_val[cols_to_scale])

X_train[cols_to_scale].describe()

,Age,Fare,Fare_per_person,SibSp,Parch,Group_Size
count,7.120000e+02,7.120000e+02,7.120000e+02,7.120000e+02,7.120000e+02,7.120000e+02
mean,7.734138e-17,-1.746418e-17,2.494883e-18,-5.613487e-18,-1.621674e-17,-4.989766e-18
std,1.000703e+00,1.000703e+00,1.000703e+00,1.000703e+00,1.000703e+00,1.000703e+00
min,-2.238460e+00,-6.625632e-01,-8.199450e-01,-4.650843e-01,-4.661832e-01,-5.761506e-01
25%,-5.805160e-01,-4.981542e-01,-4.625511e-01,-4.650843e-01,-4.661832e-01,-5.761506e-01
50%,-8.113533e-02,-3.615930e-01,-4.182722e-01,-4.650843e-01,-4.661832e-01,-5.761506e-01
75%,4.950731e-01,-1.707070e-02,2.217590e-01,4.783345e-01,-4.661832e-01,1.576942e-01
max,3.875496e+00,1.000533e+01,9.382794e+00,7.082266e+00,6.697610e+00,3.826918e+00


# 2. Save Files into Processed

In [ ]:
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

X_train.to_csv(f'{output_dir}/X_train.csv', index=False)
X_val.to_csv(f'{output_dir}/X_val.csv', index=False)
y_train.to_csv(f'{output_dir}/y_train.csv', index=False)
y_val.to_csv(f'{output_dir}/y_val.csv', index=False)

print(f"Saved to {output_dir}/")

Saved to data/processed/
